# Stage 5 - adversarial training under two threat models (decisive, averaged)

Both datasets are evaluated the SAME way, under BOTH threat models, and averaged over repeats.

- **transfer**: attack crafted from the *baseline* CNN, fed to every model (a real attacker has no access to the defended model).
- **white-box**: attack crafted against the model it hits (worst case; the Random Forest has no gradients, so white-box does not apply to it).

The metric is robust-support macro-F1 (classes with >=2 test frames, computed the same way for both datasets). CICIoV single runs are unstable on tiny data, so the averaged table below is the one to read; the first cell just shows one run in detail.

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## One run in detail (see the training and a single 2x2)

In [ ]:
# ONE illustrative run
def one_run(name, seed=42, quiet_train=False):
    d = data[name]; Xtr, ytr, Xte, yte, classes, cfg, rs = (d['Xtr'], d['ytr'], d['Xte'], d['yte'], d['classes'], d['cfg'], d['rs'])
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    ctx = contextlib.redirect_stdout(io.StringIO()) if quiet_train else contextlib.nullcontext()
    with ctx:
        base = train_cnn(CNN1D(Xtr.shape[1], len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
        defended = adversarial_train_cnn(Xtr, ytr, strategy='pgd', n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
    rf = build_random_forest(random_seed=seed).fit(Xtr, ytr)
    bcf = atk.wrap_cnn_for_art(base, Xtr.shape[1], len(classes), DEVICE)
    dcf = atk.wrap_cnn_for_art(defended, Xtr.shape[1], len(classes), DEVICE)
    Xt = atk.generate_pgd(bcf, Xte, 0.10)   # transfer (from baseline)
    Xw = atk.generate_pgd(dcf, Xte, 0.10)   # white-box (from defended)
    def f(clf, X): return mf1(yte, clf.predict(X).argmax(1), rs)
    return dict(base=(f(bcf, Xte), f(bcf, Xt), f(bcf, Xt)),
                defended=(f(dcf, Xte), f(dcf, Xt), f(dcf, Xw)),
                rf=(mf1(yte, rf.predict(Xte), rs), mf1(yte, rf.predict(Xt), rs), None))

for name in DATASETS:
    print('########## ' + name + ' ##########')
    r = one_run(name, seed=42, quiet_train=False)
    print(name + ': robust-support macro-F1, PGD eps=0.10')
    print(f"  {'model':15s}{'clean':>10}{'transfer':>10}{'white-box':>10}")
    for lab, key in [('baseline CNN','base'), ('defended CNN','defended'), ('Random Forest','rf')]:
        c, t, w = r[key]
        ws = 'n/a' if w is None else f'{w:.3f}'
        print(f"  {lab:15s}{c:>10.3f}{t:>10.3f}{ws:>10}")

## The decisive result - averaged over repeats (mean +/- std)

In [ ]:
# AVERAGED decisive result: repeat the whole thing and report mean +/- std.
# CICIoV single runs are unstable (CUDA nondeterminism on tiny data), so only the average is trustworthy.
N_REPEATS = 10
keys = ['base_clean','base_atk','def_clean','def_transfer','def_whitebox','rf_clean','rf_atk']
agg = {name: {k: [] for k in keys} for name in DATASETS}
for r in range(N_REPEATS):
    for name in DATASETS:
        res = one_run(name, seed=42 + r, quiet_train=True)
        A = agg[name]
        A['base_clean'].append(res['base'][0]); A['base_atk'].append(res['base'][1])
        A['def_clean'].append(res['defended'][0]); A['def_transfer'].append(res['defended'][1]); A['def_whitebox'].append(res['defended'][2])
        A['rf_clean'].append(res['rf'][0]); A['rf_atk'].append(res['rf'][1])
    print(f'repeat {r + 1}/{N_REPEATS} done')

def ms(v): return f'{np.mean(v):.3f}+/-{np.std(v):.3f}'
print()
print('robust-support macro-F1, mean +/- std over %d repeats, PGD eps=0.10' % N_REPEATS)
print(f"  {'dataset':12s}{'model':15s}{'clean':>16}{'transfer':>16}{'white-box':>16}")
for name in DATASETS:
    A = agg[name]
    print(f"  {name:12s}{'baseline CNN':15s}{ms(A['base_clean']):>16}{ms(A['base_atk']):>16}{ms(A['base_atk']):>16}")
    print(f"  {'':12s}{'defended CNN':15s}{ms(A['def_clean']):>16}{ms(A['def_transfer']):>16}{ms(A['def_whitebox']):>16}")
    print(f"  {'':12s}{'Random Forest':15s}{ms(A['rf_clean']):>16}{ms(A['rf_atk']):>16}{'n/a':>16}")

## Save the decisive defence result (for report writing)
Writes `results/<name>_defence_results.json`: robust-support macro-F1 (mean ± std over `N_REPEATS`) for baseline CNN, defended CNN (transfer + white-box) and Random Forest, under PGD eps=0.10.

In [4]:
import json

def ms(v):
    arr = np.array(v, dtype=float)
    return {'mean': float(arr.mean()), 'std': float(arr.std()), 'runs': [float(x) for x in arr]}

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    A, d = agg[name], data[name]
    report = {
        'dataset': name,
        'method': 'two_threat_model_averaged',
        'description': (
            'Robust-support macro-F1 under PGD eps=0.10, averaged over N_REPEATS seeded runs. '
            'transfer = attack crafted from the baseline CNN, fed to every model. '
            'white_box = attack crafted against the model it hits (CNN-defended only; '
            'the Random Forest has no gradients so white-box does not apply to it).'
        ),
        'attack_eps': 0.10,
        'n_repeats': N_REPEATS,
        'robust_support_classes': [d['classes'][i] for i in d['rs']],
        'results': {
            'baseline_cnn': {'clean': ms(A['base_clean']), 'attack': ms(A['base_atk'])},
            'defended_cnn': {'clean': ms(A['def_clean']), 'transfer': ms(A['def_transfer']), 'white_box': ms(A['def_whitebox'])},
            'random_forest': {'clean': ms(A['rf_clean']), 'attack': ms(A['rf_atk'])},
        },
    }
    path = config.RESULTS_DIR / f'{name}_defence_results.json'
    path.write_text(json.dumps(report, indent=2))
    print('saved ->', path)

saved -> /home/koala/lab/adversec/results/ciciov2024_defence_results.json
saved -> /home/koala/lab/adversec/results/road_defence_results.json


## Is the defence effect actually significant? (paired t-test across repeats)

Comparing means alone (e.g. "0.192 vs 0.327") doesn't say whether that gap is real or just
noise across repeats. Each repeat trains baseline and defended models from the *same* seed,
so `defended[r] - baseline[r]` is a proper paired comparison — a paired t-test on those
per-repeat differences is the right test here, and is much more powerful than comparing two
independent means with overlapping standard deviations.

In [ ]:
import json
from scipy import stats as sstats

sig_results = {}
for name in DATASETS:
    d = json.loads((config.RESULTS_DIR / f'{name}_defence_results.json').read_text())
    r = d['results']
    base_atk = r['baseline_cnn']['attack']['runs']
    def_transfer = r['defended_cnn']['transfer']['runs']
    def_whitebox = r['defended_cnn']['white_box']['runs']

    t_transfer, p_transfer = sstats.ttest_rel(def_transfer, base_atk)
    t_whitebox, p_whitebox = sstats.ttest_rel(def_whitebox, base_atk)
    sig_results[name] = {
        'n_repeats': len(base_atk),
        'transfer_vs_undefended': {
            'mean_diff': float(np.mean(def_transfer) - np.mean(base_atk)),
            't_statistic': float(t_transfer),
            'p_value': float(p_transfer),
            'significant_at_0.05': bool(p_transfer < 0.05),
        },
        'white_box_vs_undefended': {
            'mean_diff': float(np.mean(def_whitebox) - np.mean(base_atk)),
            't_statistic': float(t_whitebox),
            'p_value': float(p_whitebox),
            'significant_at_0.05': bool(p_whitebox < 0.05),
        },
    }

print("paired t-test (defended vs undefended, same seed per repeat) -- computed from the saved runs, no retraining needed")
print(f"  {'dataset':12s}{'comparison':22s}{'n':>4}{'mean diff':>12}{'t':>8}{'p':>10}   significant?")
for name in DATASETS:
    s = sig_results[name]
    for label, key in [('transfer', 'transfer_vs_undefended'), ('white-box', 'white_box_vs_undefended')]:
        v = s[key]
        print(f"  {name:12s}{label:22s}{s['n_repeats']:>4}{v['mean_diff']:>+12.3f}{v['t_statistic']:>8.2f}{v['p_value']:>10.4f}   "
              f"{'YES' if v['significant_at_0.05'] else 'no'}")

## Save the significance results

Merges into the `<name>_defence_results.json` already written above, adding a
`paired_significance_vs_undefended` section — doesn't touch anything already there.

In [ ]:
for name in DATASETS:
    path = config.RESULTS_DIR / f'{name}_defence_results.json'
    existing = json.loads(path.read_text())
    existing['paired_significance_vs_undefended'] = sig_results[name]
    path.write_text(json.dumps(existing, indent=2))
    print('updated ->', path)